In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import ex1_parametric
import os
import CoRT_builder
import SI_CoRT
import datetime
import json

In [ ]:
n_source = 20
p = 100
K = 5
Ka = 2
h = 15
alpha = 0.05
T = 3
s_len = 10
s_vector = [0.7] * s_len

n_target_list = [30, 40, 50, 60]

for n_target in n_target_list:
    N = n_target + Ka * n_source
    NI = n_target + n_source

    lambda_constant = 2
    lamda_k_source = lambda_constant * np.sqrt(np.log(p)/ N)
    lamda_1_source = lambda_constant * np.sqrt(np.log(p)/ NI) 
    lamda_not_source = lambda_constant * np.sqrt(np.log(p) / n_target) 
    para_results_storage = []
    CoRT_model = CoRT_builder.CoRT(alpha=lamda_not_source)
    iteration = 1

    cnt1 = 0
    cnt2 = 0
    cnt3 = 0
    cnt4 = 0

    for i in range(iteration):
        target_data, source_data = CoRT_model.gen_data(n_target, n_source, p, K, Ka, h, s_vector, s_len, "AR")
        result_dict = SI_CoRT.SI_parametric(n_target, p, K, target_data, source_data, lamda_not_source, lamda_1_source, lamda_k_source, T, s_len)
        if result_dict != None:
            cnt1 += (result_dict['is_signal'] == True)
            cnt2 += (result_dict['is_signal'] == False)
            cnt3 += (result_dict['is_signal'] == True and result_dict['p_value'] <= alpha)
            cnt4 += (result_dict['is_signal'] == False and result_dict['p_value'] <= alpha)
            if i % 100 == 0:
                print(f"is_signal : {result_dict['is_signal']}, p_values[{i}]: {result_dict['p_value']}")
                print(f"FPR: {cnt4 / cnt2}, TPR: {cnt3 / cnt1}")
                print(f"is_not_signal: {int(cnt2), int(cnt4)}")
                print(f"is_signal: {int(cnt1), int(cnt3)}")
                # print("\n")
                print("===========================================================================")
                
            para_results_storage.append(result_dict)

    is_signal_cases = [r for r in para_results_storage if r['is_signal']]
    not_signal_cases = [r for r in para_results_storage if not r['is_signal']]

    false_positives = sum(1 for c in not_signal_cases if c['p_value'] <= alpha)
    fpr = false_positives / len(not_signal_cases) if len(not_signal_cases) != 0 else -1
    true_positives = sum(1 for r in is_signal_cases if r['p_value'] <= alpha)
    tpr = true_positives / len(is_signal_cases) if len(is_signal_cases) != 0 else -1

    configs = {
        "n_source": n_source,
        "p":  p,
        "K": K,
        "Ka": Ka,
        "h": h,
        "alpha": alpha,
        "T":  T,
        "s_len": s_len,
        "s_vector": [0.7] * s_len,
        "n_target": n_target,
        "lambda_constant": lambda_constant,
        "iteration": iteration,
    }

    results = {
        "configs": configs,
        "time": datetime.datetime.now().ctime(),
        "fpr": fpr,
        "tpr": tpr
    }

    filename = 'run_parametric_vis_records.json'
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            try:
                records = json.load(f)
            except Exception as e:
                print(e)
                records = []
    else:
        records = []

    records.append(results)

    with open(filename, 'w') as f:
        json.dump(records, f, indent=4)

is_signal : True, p_values[0]: 0.03263342718146123
FPR: nan, TPR: 1.0
is_not_signal: (0, 0)
is_signal: (1, 1)
is_signal : True, p_values[0]: 0.007010846530817849
FPR: nan, TPR: 1.0
is_not_signal: (0, 0)
is_signal: (1, 1)
is_signal : True, p_values[0]: 0.0007995667682867413
FPR: nan, TPR: 1.0
is_not_signal: (0, 0)
is_signal: (1, 1)
is_signal : True, p_values[0]: 2.824909930421171e-07
FPR: nan, TPR: 1.0
is_not_signal: (0, 0)
is_signal: (1, 1)
